# 05 · Consultas con Spark SQL

**Objetivos de aprendizaje**

- Registrar un DataFrame como vista temporal y consultarlo con SQL.
- Aplicar agregaciones, filtros de grupos (`HAVING`) y clasificaciones (`CASE WHEN`).
- Usar funciones de ventana: `RANK`, `LAG`, sumas acumuladas y porcentajes del total.
- Comparar una UDF de Python con una función nativa.
- Verificar que SQL y la API de DataFrames generan el mismo plan de ejecución.

| Aspecto | Detalle |
|---|---|
| Duración estimada | 35 minutos |
| Requisito previo | Cuaderno 04 (recomendado) |
| Entrada | Dataset activo (versión limpia, si existe) |
| Salida | Respuestas a preguntas de análisis |

### Cómo utilizar este cuaderno

1. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con `Shift + Enter`. Cada celda puede depender de variables creadas en celdas anteriores.
2. Las celdas que comienzan con `# >>> CELDA EDITABLE` son las únicas que deben modificarse para adaptar el análisis a otro dataset. El resto del código se ajusta automáticamente a las columnas detectadas.
3. Cada sección sigue la misma estructura: **fundamento** (qué es y por qué se hace), **código**, **interpretación** del resultado y **adaptación** al dataset propio.
4. Ante un error `NameError` (variable no definida), lo habitual es que no se haya ejecutado una celda anterior. La solución es *Kernel > Restart Kernel and Run All Cells*.
5. Mientras la sesión de Spark está activa, la Spark UI (normalmente <http://localhost:4040>) muestra los *jobs*, *stages* y tareas ejecutados.

La guía completa de uso de los cuadernos se encuentra en `docs/08-uso-de-notebooks.md`.

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("05_SparkSQL")

## Carga del dataset activo

El dataset activo es el que se configuró en el cuaderno `01_configuracion_dataset.ipynb` y quedó guardado en `data/dataset_config.json`. Si no se ha configurado ninguno, se utiliza el dataset de ejemplo (`data/ejemplo/ventas_tienda.csv`).

La función `clasificar_columnas()` asigna a cada columna un rol, que los cuadernos usan para decidir qué análisis aplicar:

| Variable | Rol | Criterio |
|---|---|---|
| `NUM` | numéricas | tipos numéricos que no son identificadores |
| `CAT` | categóricas | texto o booleanos con pocos valores distintos (hasta `max_categorias`) |
| `FEC` | fechas | tipos `date` o `timestamp` |
| `TXT` | texto | texto con muchos valores distintos (comentarios, descripciones) |
| — | identificadores | columnas excluidas en la configuración o con valores casi únicos |

Con un dataset propio, conviene revisar esta clasificación. Si una columna quedó en un rol inadecuado, puede corregirse en la configuración (`columnas_excluir`, `max_categorias`) o directamente en las listas, por ejemplo `CAT.remove("columna")`.

In [ ]:
# Carga del dataset activo (definido y guardado en el cuaderno 01)
df, cfg = cargar_desde_config(spark, version="auto")

# Clasificación automática de columnas por rol analítico
roles = clasificar_columnas(df, excluir=cfg["columnas_excluir"], max_categorias=cfg["max_categorias"])
NUM, CAT, FEC, TXT = roles["numericas"], roles["categoricas"], roles["fechas"], roles["texto"]
objetivo = columna_objetivo(cfg, df)
resumen_roles(roles)
print("Columna objetivo:", objetivo)

## 1. Fundamento: Spark SQL

Spark SQL permite consultar datos distribuidos con SQL estándar. Las consultas SQL y las operaciones con DataFrames pasan por el mismo optimizador, Catalyst, que transforma la consulta en un plan físico eficiente (Armbrust et al., 2015). Para consultar un DataFrame con SQL, primero se registra como **vista temporal**, que existe mientras dure la sesión.

In [ ]:
tabla = cfg["nombre_tabla"]
df.createOrReplaceTempView(tabla)
spark.sql(f"DESCRIBE {tabla}").show(50, truncate=False)

## 2. Columnas para las consultas de ejemplo

Las consultas de este cuaderno se construyen con las columnas elegidas en la celda siguiente. Por defecto se toman de la clasificación automática: `CAT1` y `CAT2` son categóricas, `NUM1` es la variable numérica principal (la objetivo, si es numérica) y `FECHA` es la fecha principal.

**Adaptación.** Con un dataset propio, conviene elegir columnas que respondan preguntas de interés; por ejemplo, `CAT1 = "departamento"`, `NUM1 = "valor_total"`.

In [ ]:
# >>> CELDA EDITABLE: columnas para las consultas
CAT1 = CAT[0] if CAT else None
CAT2 = CAT[1] if len(CAT) > 1 else None
NUM1 = (objetivo if objetivo in NUM else NUM[0]) if NUM else None
FECHA = FEC[0] if FEC else None
print(f"CAT1={CAT1} | CAT2={CAT2} | NUM1={NUM1} | FECHA={FECHA}")

Las consultas usan *f-strings* de Python para insertar los nombres de columna en el texto SQL, y comillas invertidas (`` ` ``) alrededor de cada nombre, lo que permite usar columnas con nombres reservados.

## 3. Agregación por grupos

`GROUP BY` resume los registros por cada valor de una columna. Las funciones `COUNT`, `AVG`, `SUM`, `MIN` y `MAX` se calculan de forma distribuida en cada partición y luego se combinan.

In [ ]:
if CAT1 and NUM1:
    spark.sql(f"""
        SELECT `{CAT1}`,
               COUNT(*)                 AS registros,
               CAST(AVG(`{NUM1}`) AS DECIMAL(22,2)) AS promedio,
               CAST(SUM(`{NUM1}`) AS DECIMAL(22,2)) AS suma
        FROM {tabla}
        GROUP BY `{CAT1}`
        ORDER BY suma DESC
    """).show(15, truncate=False)

## 4. Filtro de grupos con HAVING

`WHERE` filtra filas **antes** de agrupar; `HAVING` filtra los grupos **después** de agrupar. La consulta siguiente conserva solo los grupos con más registros que el promedio de registros por grupo; este último se calcula en una subconsulta.

In [ ]:
if CAT1:
    spark.sql(f"""
        SELECT `{CAT1}`, COUNT(*) AS registros
        FROM {tabla}
        GROUP BY `{CAT1}`
        HAVING COUNT(*) > (SELECT AVG(n) FROM (SELECT COUNT(*) AS n FROM {tabla} GROUP BY `{CAT1}`))
        ORDER BY registros DESC
    """).show(truncate=False)

## 5. Clasificación con CASE WHEN

`CASE WHEN` crea categorías a partir de condiciones. Los cortes se calculan con los percentiles 33 y 66 de `NUM1`, de modo que cada rango agrupe aproximadamente un tercio de los registros.

In [ ]:
if NUM1:
    p33, p66 = df.approxQuantile(NUM1, [0.33, 0.66], 0.01)
    print(f"Cortes: p33 = {p33:,.2f} | p66 = {p66:,.2f}")
    spark.sql(f"""
        SELECT CASE WHEN `{NUM1}` <  {p33} THEN '1. Bajo'
                    WHEN `{NUM1}` <  {p66} THEN '2. Medio'
                    ELSE '3. Alto' END            AS rango,
               COUNT(*)                            AS registros,
               CAST(AVG(`{NUM1}`) AS DECIMAL(22,2)) AS promedio
        FROM {tabla}
        WHERE `{NUM1}` IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """).show()

## 6. Funciones de ventana

Una función de ventana calcula un valor para cada fila a partir de un grupo de filas relacionadas (la ventana), **sin colapsar** las filas como lo hace `GROUP BY`. Se define con `OVER (PARTITION BY ... ORDER BY ...)` (ASF, 2026b).

### 6.1 Ranking dentro de cada grupo

Se obtienen los tres valores de `CAT2` con mayor suma de `NUM1` dentro de cada valor de `CAT1`.

In [ ]:
if CAT1 and CAT2 and NUM1:
    spark.sql(f"""
        WITH resumen AS (
            SELECT `{CAT1}`, `{CAT2}`, CAST(SUM(`{NUM1}`) AS DECIMAL(22,2)) AS suma
            FROM {tabla}
            GROUP BY `{CAT1}`, `{CAT2}`
        )
        SELECT * FROM (
            SELECT *, RANK() OVER (PARTITION BY `{CAT1}` ORDER BY suma DESC) AS posicion
            FROM resumen
        )
        WHERE posicion <= 3
        ORDER BY `{CAT1}`, posicion
    """).show(30, truncate=False)

### 6.2 Participación porcentual

`SUM(...) OVER ()`, con una ventana vacía, calcula el total general en cada fila; así se obtiene el porcentaje que aporta cada grupo.

In [ ]:
if CAT1 and NUM1:
    spark.sql(f"""
        SELECT `{CAT1}`,
               CAST(SUM(`{NUM1}`) AS DECIMAL(22,2))                       AS suma,
               ROUND(100 * SUM(`{NUM1}`) / SUM(SUM(`{NUM1}`)) OVER (), 2) AS porcentaje
        FROM {tabla}
        GROUP BY `{CAT1}`
        ORDER BY porcentaje DESC
    """).show(15, truncate=False)

### 6.3 Evolución temporal: LAG y suma acumulada

`LAG()` devuelve el valor de la fila anterior dentro de la ventana, lo que permite calcular la variación entre periodos. `SUM() OVER (ORDER BY ...)` produce la suma acumulada.

In [ ]:
if FECHA and NUM1:
    spark.sql(f"""
        WITH mensual AS (
            SELECT DATE_TRUNC('month', `{FECHA}`) AS periodo, SUM(`{NUM1}`) AS suma
            FROM {tabla}
            WHERE `{FECHA}` IS NOT NULL
            GROUP BY 1
        )
        SELECT DATE_FORMAT(periodo, 'yyyy-MM')                                         AS mes,
               CAST(suma AS DECIMAL(22,2))                                             AS suma,
               ROUND(100 * (suma - LAG(suma) OVER (ORDER BY periodo)) / LAG(suma) OVER (ORDER BY periodo), 1) AS variacion_pct,
               CAST(SUM(suma) OVER (ORDER BY periodo) AS DECIMAL(22,2))                AS acumulado
        FROM mensual
        ORDER BY periodo
    """).show(36)
else:
    print("Se requiere una columna de fecha y una numérica.")

## 7. Funciones definidas por el usuario (UDF)

Una UDF permite aplicar una función de Python dentro de una consulta. Su desventaja es el rendimiento: cada valor debe enviarse desde la JVM al intérprete de Python y regresar. Por eso solo se recomiendan cuando no existe una función nativa equivalente (Damji et al., 2020). La comparación siguiente aplica la misma clasificación con una UDF y con `CASE WHEN`.

In [ ]:
import time
from pyspark.sql.types import StringType

if NUM1:
    def rango_python(valor):
        if valor is None:
            return None
        return "1. Bajo" if valor < p33 else ("2. Medio" if valor < p66 else "3. Alto")

    spark.udf.register("rango_python", rango_python, StringType())

    t = time.time()
    spark.sql(f"SELECT rango_python(`{NUM1}`) AS r, COUNT(*) FROM {tabla} GROUP BY 1").collect()
    t_udf = time.time() - t

    t = time.time()
    spark.sql(f"""SELECT CASE WHEN `{NUM1}` < {p33} THEN '1. Bajo' WHEN `{NUM1}` < {p66} THEN '2. Medio'
                  ELSE '3. Alto' END AS r, COUNT(*) FROM {tabla} GROUP BY 1""").collect()
    print(f"UDF de Python: {t_udf:.2f} s | CASE WHEN nativo: {time.time() - t:.2f} s")

## 8. SQL y DataFrames: el mismo plan

Las dos consultas siguientes, una en SQL y otra con la API de DataFrames, producen el mismo plan físico, lo que demuestra que la elección entre ambas es de estilo y no de rendimiento (Armbrust et al., 2015).

In [ ]:
if CAT1 and NUM1:
    spark.sql(f"SELECT `{CAT1}`, SUM(`{NUM1}`) AS s FROM {tabla} GROUP BY `{CAT1}`").explain()
    df.groupBy(CAT1).agg(F.sum(NUM1).alias("s")).explain()

## 9. Consultas propias

La celda siguiente es una plantilla para responder las preguntas de análisis del dataset propio. Se recomienda escribir primero la pregunta en una celda Markdown y luego la consulta que la responde.

**Pregunta 1** (doble clic para editar): ...

In [ ]:
# >>> CELDA EDITABLE: consulta propia
consulta = f"""
SELECT *
FROM {tabla}
LIMIT 10
"""
spark.sql(consulta).show(truncate=False)

## Ejercicios propuestos

1. Determinar el valor de `CAT1` con el mayor promedio de `NUM1` y el de menor promedio.
2. Con `ROW_NUMBER()`, obtener para cada valor de `CAT2` el registro con el mayor `NUM1`.
3. Calcular qué porcentaje de los registros tiene `NUM1` por encima del promedio general.
4. Formular tres preguntas de negocio sobre el dataset propio y responderlas con SQL.

In [ ]:
spark.stop()

## Referencias

Armbrust, M., Xin, R. S., Lian, C., Huai, Y., Liu, D., Bradley, J. K., Meng, X., Kaftan, T., Franklin, M. J., Ghodsi, A., & Zaharia, M. (2015). Spark SQL: Relational data processing in Spark. En *Proceedings of the 2015 ACM SIGMOD International Conference on Management of Data* (pp. 1383–1394). ACM. https://doi.org/10.1145/2723372.2742797

Damji, J. S., Wenig, B., Das, T., & Lee, D. (2020). *Learning Spark: Lightning-fast data analytics* (2.ª ed.). O'Reilly Media.

The Apache Software Foundation. (2026b). *Spark SQL, DataFrames and Datasets guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/sql-programming-guide.html